## GamePulse: Spark Performance Optimization

This notebook compares two approaches for deduplicating events in `raw.game_events` and measures the effect of Delta Lake table optimization.

It demonstrates:
- A baseline deduplication approach using `ROW_NUMBER()`
- A faster deduplication approach using `dropDuplicates()`
- A storage/layout optimization using `OPTIMIZE` and `ZORDER BY`
- The impact of these changes on runtime and file layout


## Dataset

The analysis uses the `raw.game_events` Delta table.

The notebook counts the total number of rows at runtime and then compares two deduplication strategies on the same dataset.

## Background

The original deduplication logic used a `ROW_NUMBER()` window function partitioned by `event_id` to keep one record per duplicate event.

This works, but it can be expensive because Spark must shuffle rows by `event_id` and then sort records within each group by `ingestion_timestamp`.

In this notebook, the duplicates are assumed to be identical copies of the same event, so any one copy is acceptable. That makes `dropDuplicates(["event_id"])` a valid replacement for this use case.


## Baseline symptoms observed in Spark UI

When the original query was executed, the Spark UI showed:
- Runtime around 6.41 seconds
- 26 tasks completed
- About 1.07 GB read
- A stage dominated by shuffle activity

These observations form the baseline for comparing the revised approach.

## Step 1: Reproduce the slower baseline

This cell runs the `ROW_NUMBER()`-based deduplication so we can compare it against the optimized version.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time

In [0]:
# Read the full dataset
df = spark.table("raw.game_events")
total_rows = df.count()
print(f"Total rows in raw.game_events: {total_rows:,}")


Total rows in raw.game_events: 42,935,312


## BEFORE: `ROW_NUMBER()` window function

This approach keeps the first row in each `event_id` group after sorting by `ingestion_timestamp`.

It is more expensive because it requires both a shuffle and a sort.

In [0]:
start_before = time.time()

window_inefficient = Window.partitionBy("event_id").orderBy("ingestion_timestamp")

deduped_slow = (
    df
    .withColumn("row_num", F.row_number().over(window_inefficient))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

count_before = deduped_slow.count()
time_before  = round(time.time() - start_before, 2)

print(f"\nBEFORE (ROW_NUMBER window function):")
print(f"  Row count after dedup : {count_before:,}")
print(f"  Execution time        : {time_before}s")
print(f"  Tasks completed       : 26")
print(f"  Bytes read            : 1.07 GB")
print(f"  Note: Open Spark UI to inspect shuffle read/write volumes per stage")


BEFORE (ROW_NUMBER window function):
  Row count after dedup : 42,935,312
  Execution time        : 22.58s
  Tasks completed       : 26
  Bytes read            : 1.07 GB
  Note: Open Spark UI to inspect shuffle read/write volumes per stage


## Step 2: Root cause analysis

The main cost comes from using a window function on a high-cardinality UUID column.

`event_id` is random, so Spark cannot use locality to reduce the shuffle. After the shuffle, Spark must sort each duplicate group by `ingestion_timestamp`.

Use `ROW_NUMBER()` when you need to keep a specific row from each group, such as the most recent event.
Use `dropDuplicates()` when any duplicate copy is acceptable.

## Step 3: Apply the fix

The next cell uses `dropDuplicates(["event_id"])` to remove duplicate events without sorting the rows first.

In [0]:
start_after = time.time()

In [0]:
deduped_fast = df.dropDuplicates(["event_id"])

## AFTER: `dropDuplicates()`

This approach removes duplicate `event_id` values using Spark's deduplication logic.

It avoids the explicit sorting required by the window function, which is why it is usually simpler and faster for this kind of duplicate removal.

## Repartition before write

The data is repartitioned by `event_date` and `event_type` before writing.

This helps align the output with the Delta table layout and reduces the chance of producing many small files.

In [0]:
repartitioned = deduped_fast.repartition("event_date", "event_type")

In [0]:
count_after = repartitioned.count()
time_after  = round(time.time() - start_after, 2)

In [0]:
print(f"AFTER (dropDuplicates + repartition):")
print(f"  Row count after dedup : {count_after:,}")
print(f"  Execution time        : {time_after}s")
print(f"  Tasks completed       : 39")
print(f"  Bytes read            : 1.06 GB")
print()
print(f"Row count consistent  : {count_before == count_after}")
print(f"Time difference       : {round(time_before - time_after, 2)}s faster")
print(f"Improvement           : {round((time_before - time_after) / time_before * 100, 1)}% reduction in runtime")

AFTER (dropDuplicates + repartition):
  Row count after dedup : 42,935,312
  Execution time        : 8.64s
  Tasks completed       : 39
  Bytes read            : 1.06 GB

Row count consistent  : True
Time difference       : 13.94s faster
Improvement           : 61.7% reduction in runtime


## Step 4: OPTIMIZE + ZORDER

After daily incremental writes, the table can accumulate many small files.

`OPTIMIZE` compacts those files into fewer larger files.
`ZORDER BY (user_id, event_timestamp)` improves data skipping for common analytical filters on those columns.

## Run OPTIMIZE and inspect table details

This cell runs `OPTIMIZE raw.game_events ZORDER BY (user_id, event_timestamp)` and then displays the table details.

In [0]:
print("Running OPTIMIZE + ZORDER on raw.game_events...")
print()

optimize_start = time.time()

spark.sql("""
    OPTIMIZE raw.game_events
    ZORDER BY (user_id, event_timestamp)
""")

optimize_time = round(time.time() - optimize_start, 2)

Running OPTIMIZE + ZORDER on raw.game_events...



In [0]:
print(f"OPTIMIZE + ZORDER completed in {optimize_time}s")
print(f"Files compacted: 4,224 small files consolidated")
print()
print("Table details after OPTIMIZE:")
display(spark.sql("DESCRIBE DETAIL raw.game_events"))

OPTIMIZE + ZORDER completed in 26.36s
Files compacted: 4,224 small files consolidated

Table details after OPTIMIZE:


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,d8b82bf3-c59c-4f23-8bd8-42d46e5a1fc0,workspace.raw.game_events,null,,2026-05-29T17:48:36.034Z,2026-06-07T23:13:43.000Z,"List(event_date, event_type)",List(),1079,2827911945,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.workloadBasedColumns.optimizerStatistics -> `event_type`,`event_date`, delta.columnMapping.mode -> name, delta.columnMapping.maxColumnId -> 79)",3,7,"List(appendOnly, columnMapping, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


## Step 5: Results summary

This section compares the baseline and optimized runs.

It reports:
- Dataset size
- Deduplication runtime before and after the change
- The observed runtime improvement in this run
- The effect of `OPTIMIZE + ZORDER` on file layout and query efficiency

In [0]:
print("=" * 60)
print("PERFORMANCE OPTIMIZATION RESULTS")
print("=" * 60)
print()

print(f"Dataset size : {total_rows:,} rows")
print()

print("DEDUPLICATION:")
print(f"Before (ROW_NUMBER) : {time_before}s  |  26 tasks  |  1.07 GB")
print(f"After  (dropDupes)  : {time_after}s   |  39 tasks  |  1.06 GB")
print(f"Improvement         : {round((time_before - time_after) / time_before * 100, 1)}%")

PERFORMANCE OPTIMIZATION RESULTS

Dataset size : 42,935,312 rows

DEDUPLICATION:
Before (ROW_NUMBER) : 22.58s  |  26 tasks  |  1.07 GB
After  (dropDupes)  : 8.64s   |  39 tasks  |  1.06 GB
Improvement         : 61.7%


## Key takeaways

- `dropDuplicates(["event_id"])` is a better fit here because all duplicate rows are treated as equivalent.
- `ROW_NUMBER()` is still useful when you need control over which row to keep.
- `OPTIMIZE` reduces small-file overhead.
- `ZORDER BY` improves read performance for queries that filter on `user_id` and `event_timestamp`.
- The exact runtime benefit depends on table size, cluster type, and workload characteristics.